# Figure 4.1 - composing two shallow networks

![Figure 4.1: composing two shallow networks](assets/figure-4-1-composing-networks.png)

> **Figure source:** Figure 4.1 from [*Understanding Deep Learning*](../chapter-04-deep-neural-networks.pdf), book page 42, by Simon J. D. Prince (MIT Press). The supplied screenshot is included unchanged for noncommercial study under [CC BY-NC-ND 4.0](../../../LICENSES/CC-BY-NC-ND-4.0.txt).

The diagram in panel (a) connects two shallow networks in sequence. Network 1 maps the original input to an intermediate value, and Network 2 maps that intermediate value to the final output:

$$
y=f_1(x), \qquad y'=f_2(y), \qquad y'=f_2(f_1(x)).
$$

Panel (b) plots $f_1$: horizontal position is $x$ and vertical position is $y$. Panel (c) plots $f_2$: its horizontal position is now $y$, because the output of Network 1 is the input to Network 2. Panel (d) plots the composition $f_2(f_1(x))$ directly against the original input $x$.

# Why can different values of x produce the same y?

A deterministic function only requires **each input to have one output**. It does not require different inputs to have different outputs. A function that never maps two inputs to the same output is called **one-to-one** or **injective**. Network 1 is deterministic, but it is deliberately not injective.

A familiar example is $f(x)=x^2$: both $x=-2$ and $x=2$ produce $y=4$. Nothing is inconsistent about this; the graph simply fails the horizontal-line test for a one-to-one function.

The same idea appears in panel (b). The graph rises, falls, and rises again, so a horizontal dashed line intersects it at three places. The three gray points are different values $x_a$, $x_b$, and $x_c$, but all three intersections have the same vertical coordinate $y_0$:

$$
f_1(x_a)=f_1(x_b)=f_1(x_c)=y_0.
$$

The network can make this zigzag because its output is a weighted sum of ReLU hidden units:

$$
f_1(x)=\phi_0+\sum_i \phi_i\,\operatorname{ReLU}(\theta_{i0}+\theta_{i1}x).
$$

As hidden units switch on or off, the slope changes. Positive and negative output weights can make successive regions slope upward or downward. Therefore the complete piecewise-linear function does not have to be monotonic, and repeated output values are allowed.

> **Important distinction:** the network does not produce several outputs for one $x$. Each $x$ still has exactly one $y$. Instead, several different $x$ values happen to share that same $y$.

# What does Network 2 do with those repeated values?

Network 2 never sees the original $x$. It receives only the intermediate scalar $y$. If three inputs produce the same $y_0$, Network 2 must process all three in exactly the same way:

$$
f_2(f_1(x_a))=f_2(f_1(x_b))=f_2(f_1(x_c))=f_2(y_0).
$$

This is what the colored markers show. In panel (b), the three gray $x$ values map to the cyan $y_0$. In panel (c), that cyan input maps to the brown output $y'_0$. In panel (d), the same three gray $x$ positions therefore meet the same dashed horizontal level $y'_0$.

There is an information-loss consequence: once Network 1 has compressed $x_a$, $x_b$, and $x_c$ into the same scalar $y_0$, no later deterministic network that sees only $y_0$ can tell which original input was used. If the task required different final answers for those three inputs, this particular intermediate representation would be unsuitable. A wider vector representation could preserve information that this one-dimensional bottleneck discards.

# Why does the composed graph contain a repeated pattern?

Although the three exact inputs above must share one final output, each surrounding branch of Network 1 sweeps through a range of $y$ values. Network 2 applies the same function $f_2$ every time a branch sweeps through that range. This reproduces the shape from panel (c) several times in panel (d).

## Where are the three copies in panel (d)?

Split panel (d) at the same two $x$ positions where Network 1 bends in panel (b): $x=0$ and approximately $x=0.6$. The three adjacent intervals are the three copies:

| Copy in panel (d) | Branch of Network 1 in panel (b) | Direction through panel (c) |
|---|---|---|
| 1 | $x=-1$ to $x=0$; $y$ rises from $-1$ to $+1$ | Left to right: normal orientation |
| 2 | $x=0$ to about $x=0.6$; $y$ falls from $+1$ to $-1$ | Right to left: horizontally flipped |
| 3 | About $x=0.6$ to $x=1$; $y$ rises from $-1$ to $+1$ | Left to right: normal orientation again |

Read from left to right, the graph in panel (c) follows approximately this sequence of output values:

$$
+0.6 \;\downarrow\; -0.8 \;\uparrow\; +0.3 \;\downarrow\; -0.15.
$$

You can find that whole down-up-down sequence in panel (d) from $x=-1$ to $x=0$. From $x=0$ to about $x=0.6$, the sequence appears backward (up-down-up) because Network 1 is descending. From about $x=0.6$ to $x=1$, the original down-up-down sequence appears again. The orange first segment in panel (c) is also repeated at the start of copies 1 and 3 and at the end of the reversed copy 2.

The copies are **next to one another**, not drawn on top of one another. They have different widths because the three branches in panel (b) have different slopes. A larger absolute slope makes Network 1 move through the $y$ range more quickly, so the copy is horizontally compressed; a smaller absolute slope stretches it. A negative slope reverses it. Thus 'duplicated' means that the same sequence of corners and output values from panel (c) occurs three times, not that three equal-sized shapes are visible.

- When a branch of $f_1$ rises, it traverses Network 2's input axis from left to right, so the pattern keeps its orientation.
- When a branch of $f_1$ falls, it traverses that axis from right to left, so the pattern is horizontally flipped.
- A steep or shallow branch of $f_1$ compresses or stretches the pattern along the $x$ axis.

Away from the joints, the chain rule makes this precise:

$$
\frac{dy'}{dx}=f_2'(f_1(x))\,f_1'(x).
$$

The sign of $f_1'(x)$ controls whether the pattern is flipped, while its magnitude controls the horizontal scaling. Also, every breakpoint of $f_2$ can acquire several preimages through $f_1$, creating several breakpoints in the composed function. This multiplication of linear regions is the main lesson of the figure: depth can reuse and rearrange a simple function to create a much more intricate one.

The weights in this explanatory figure were selected to demonstrate this construction; it does not claim that training will always discover it. During learning, a network may develop many-to-one intermediate mappings when doing so reduces the loss.

> **Key takeaway:** different $x$ values can share an intermediate $y$ because Network 1 is non-monotonic and many-to-one. Equal intermediate values must produce equal final values, but the neighborhoods around those inputs cause Network 2's whole piecewise-linear pattern to be repeated, flipped, and rescaled.